# 06 - Option B refined: intermediate ViT feature + multi-subject Ridge

Two upgrades over notebook 04 (the Option B winner):

1. **Intermediate ViT feature instead of in silico EEG.** BERG's
   `eeg-things_eeg_2-vit_b_32` model runs images through a ViT-B/32 backbone and
   then a small regression to EEG. Notebook 04 used the FINAL in silico EEG
   `(63, 140)` as the feature. Here we instead tap the ViT backbone directly and
   use its **last transformer layer CLS token (768-dim)** - a richer, more
   general visual representation than the EEG head's output.

2. **Train on all 4 development subjects (sub-01..04) pooled.** The same video
   produces the same ViT feature, but each subject contributes its own real EEG,
   so a video seen by k subjects contributes k training rows (same X, different
   y). This is allowed: training data of all subjects may be used. We still
   evaluate per subject using that subject's test ground truth.

Pipeline: `ViT CLS feature -> StandardScaler -> PCA(0.95) -> StandardScaler ->
RidgeCV -> real EEG`, then correlation-match predicted vs measured test EEG.

> Rule note: this is a model-selection/analysis notebook on the development
> subjects. For the held-out submission (sub-05/06) you must NOT use sub-01..04
> test recordings as matching templates - that constraint is handled in the
> submission notebook, not here.

In [1]:
# Windows UTF-8 fix + clean berg import (see notebooks 03-05)
import builtins, sys
if not getattr(builtins, '_utf8_open_patched', False):
    _orig_open = builtins.open
    def _utf8_open(*a, **kw):
        mode = kw.get('mode', a[1] if len(a) > 1 else 'r')
        if 'b' not in mode and kw.get('encoding') is None:
            kw['encoding'] = 'utf-8'
        return _orig_open(*a, **kw)
    builtins.open = _utf8_open
    builtins._utf8_open_patched = True
for _m in [m for m in list(sys.modules) if m == 'berg' or m.startswith('berg.')]:
    del sys.modules[_m]

import os, glob
import numpy as np
import cv2
import torch
from scipy.signal import decimate
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

device: cuda


In [2]:
# ---------------- Config ----------------
BERG_DIR   = './brain-encoding-response-generator'
MODEL_ID   = 'eeg-things_eeg_2-vit_b_32'

STIM_DIR        = './stimuli_train'
STIM_TEST_DIR   = './stimuli_test'
FRAME_DIR       = './frames'
FRAME_TEST_DIR  = './frames_test'
TRAIN_DIR       = './trn'
TEST_DIR        = './tst'
GT_DIR          = './tst_stim'
CACHE_DIR       = './features_vit'
os.makedirs(CACHE_DIR, exist_ok=True)

DEV_SUBJECTS = [1, 2, 3, 4]        # the 4 development subjects (have test labels)
VIT_LAYER    = 'encoder.layers.encoder_layer_11.add_1'   # last ViT transformer block
POOL         = 'cls'               # 'cls' = token 0, or 'mean' over tokens

DEC, N_FRAMES, IMG_SIZE = 5, 8, 224
tst_ids = list(range(1001, 1103))
N_TEST  = len(tst_ids)

In [3]:
# ---------------- Helpers ----------------
def video_for(stim_id, vids):
    targets = {f'{int(stim_id)}.mp4', f'{int(stim_id):04d}.mp4'}
    hits = [v for v in vids if os.path.basename(v) in targets]
    assert hits, f'no video for stimulus {stim_id} (tried {targets})'
    return hits[0]

def dump_frames(stim_ids, stim_dir, out_dir, n_frames=N_FRAMES):
    vids = sorted(glob.glob(os.path.join(stim_dir, '*.mp4')))
    os.makedirs(out_dir, exist_ok=True)
    for sid in stim_ids:
        if len(glob.glob(os.path.join(out_dir, f'{int(sid):05d}_f*.jpg'))) >= n_frames:
            continue
        cap = cv2.VideoCapture(video_for(sid, vids))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        picks = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
        for k, fno in enumerate(picks):
            path = os.path.join(out_dir, f'{int(sid):05d}_f{k:02d}.jpg')
            if os.path.exists(path):
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fno))
            ok, frame = cap.read()
            if ok:
                cv2.imwrite(path, frame)
        cap.release()

def frames_for(sid, out_dir):
    return sorted(glob.glob(os.path.join(out_dir, f'{int(sid):05d}_f*.jpg')))

def load_rgb(path):
    return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)

def zrows(A):
    return (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)

## 1. Load BERG and tap the intermediate ViT feature extractor
We reuse `eeg_model.transform` (ViT preprocessing) and `eeg_model.feature_extractor`
(the 12-layer ViT), but we stop at the chosen layer's CLS token and never run the
EEG regression head.

In [4]:
from berg import BERG
berg = BERG(berg_dir=BERG_DIR)
eeg_model = berg.get_encoding_model(MODEL_ID, subject=1, device=DEVICE)

_transform = eeg_model.transform              # ViT_B_32 ImageNet preprocessing
_vit = eeg_model.feature_extractor            # returns dict of 12 layer outputs

def vit_feature(rgb_images):
    """list of RGB frames -> (n, 768) ViT embeddings from VIT_LAYER."""
    stim = np.stack([
        np.transpose(cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA), (2, 0, 1))
        for im in rgb_images
    ]).astype('uint8')                         # (n, 3, 224, 224)
    x = _transform(torch.from_numpy(stim)).to(DEVICE)
    with torch.no_grad():
        feats = _vit(x)[VIT_LAYER]             # (n, 50 tokens, 768)
    if POOL == 'cls':
        emb = feats[:, 0, :]                    # CLS token
    else:
        emb = feats.mean(dim=1)                 # mean over tokens
    return emb.detach().cpu().numpy()           # (n, 768)

# probe
_p = vit_feature([load_rgb(frames_for(tst_ids[0], FRAME_TEST_DIR)[0])]) if frames_for(tst_ids[0], FRAME_TEST_DIR) else None
print('ViT feature dim:', None if _p is None else _p.shape)

c:\Users\Tharun's Alien\Desktop\FruitBasket-SchubKI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1018 11:20:04.567000 3984 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Model loaded on cuda for subject 1
ViT feature dim: (1, 768)


## 2. Per-video ViT features (cached)
One 768-d vector per video = mean of its 8 frames' ViT embeddings. Videos are
shared across subjects, so we cache a single id->feature table and reuse it.

In [5]:
def build_vit_features(ids, stim_dir, frame_dir, cache_name):
    cache_path = os.path.join(CACHE_DIR, cache_name)
    if os.path.exists(cache_path):
        d = np.load(cache_path)
        if np.array_equal(d['ids'], np.asarray(ids)):
            return d['X']
    dump_frames(ids, stim_dir, frame_dir)
    rows = []
    for n, sid in enumerate(ids):
        paths = frames_for(sid, frame_dir)
        assert paths, f'no frames for stimulus {sid} in {frame_dir}'
        emb = vit_feature([load_rgb(p) for p in paths])   # (n_frames, 768)
        rows.append(emb.mean(axis=0))
        if (n + 1) % 100 == 0 or n + 1 == len(ids):
            print(f'  {n+1}/{len(ids)}')
    X = np.stack(rows)
    np.savez_compressed(cache_path, X=X, ids=np.asarray(ids))
    return X

# Test features: same 102 videos for everyone
X_tst = build_vit_features(tst_ids, STIM_TEST_DIR, FRAME_TEST_DIR, 'vit_tst_shared.npz')
print('test ViT features:', X_tst.shape)

# A lookup from a video id -> its ViT feature, built once over the union of all
# dev-subject training ids (so each video is encoded only once).
all_trn_ids = set()
trn_cache = {}
for s in DEV_SUBJECTS:
    t = np.load(f'{TRAIN_DIR}/sub-{s:02d}_trn.npz')
    ids = np.array(sorted(t.files, key=int), dtype=int)
    trn_cache[s] = (ids, t)
    all_trn_ids.update(ids.tolist())
all_trn_ids = np.array(sorted(all_trn_ids))
print('unique training videos across dev subjects:', len(all_trn_ids))

X_union = build_vit_features(all_trn_ids, STIM_DIR, FRAME_DIR, 'vit_trn_union.npz')
feat_of = {int(i): X_union[k] for k, i in enumerate(all_trn_ids)}   # id -> 768-d
print('union feature table:', X_union.shape)

  100/102
  102/102
test ViT features: (102, 768)
unique training videos across dev subjects: 1000
  100/1000
  200/1000
  300/1000
  400/1000
  500/1000
  600/1000
  700/1000
  800/1000
  900/1000
  1000/1000
union feature table: (1000, 768)


## 3. Pool training pairs across all 4 dev subjects
For each subject and each of its training videos, pair the video's ViT feature
(X) with that subject's real decimated EEG (y). Stack all pairs into one big set.

In [6]:
X_parts, Y_parts = [], []
n_chan = n_time = None
for s in DEV_SUBJECTS:
    ids, t = trn_cache[s]
    eeg = np.stack([t[str(i)] for i in ids])
    eeg = decimate(eeg, DEC, axis=-1, ftype='fir', zero_phase=True)
    n_chan, n_time = eeg.shape[1], eeg.shape[2]
    X_parts.append(np.stack([feat_of[int(i)] for i in ids]))
    Y_parts.append(eeg.reshape(len(ids), -1))
    print(f'sub-{s:02d}: {len(ids)} training pairs')

X_trn = np.concatenate(X_parts, axis=0)      # (sum_k n_k, 768)
Y_trn = np.concatenate(Y_parts, axis=0)      # (sum_k n_k, 17*300)
print('pooled training set:', X_trn.shape, '->', Y_trn.shape)

sub-01: 968 training pairs
sub-02: 972 training pairs
sub-03: 961 training pairs
sub-04: 947 training pairs
pooled training set: (3848, 768) -> (3848, 5100)


## 4. Train one shared encoder on the pooled data

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV

sc  = StandardScaler().fit(X_trn)
pca = PCA(n_components=0.95, svd_solver='full').fit(sc.transform(X_trn))
X_trn_p = pca.transform(sc.transform(X_trn))
sc2 = StandardScaler().fit(X_trn_p)
print('PCA components:', X_trn_p.shape[1])

model = RidgeCV(alphas=np.logspace(-3, 6, 20))
model.fit(sc2.transform(X_trn_p), Y_trn)
print('alpha =', model.alpha_)

# Precompute predicted EEG for the 102 test videos (shared across subjects)
X_tst_s = sc2.transform(pca.transform(sc.transform(X_tst)))
pred_tst = model.predict(X_tst_s).reshape(N_TEST, n_chan, n_time)
print('predicted test EEG:', pred_tst.shape)

PCA components: 389
alpha = 12742.749857031322
predicted test EEG: (102, 17, 300)


## 5. Evaluate per development subject (validation order) and report

In [8]:
P = zrows(pred_tst[:, :, 5:60].reshape(N_TEST, -1))     # predicted, video order
video_ids = np.array(tst_ids)

results = {}
for s in DEV_SUBJECTS:
    tst_eeg = np.load(f'{TEST_DIR}/sub-{s:02d}_tst.npy')
    tst_eeg = decimate(tst_eeg, DEC, axis=-1, ftype='fir', zero_phase=True)
    gt = np.load(f'{GT_DIR}/sub-{s:02d}_tst_stim.npy')
    order = np.argsort(gt)                               # label-sorted (validation only)

    M = zrows(tst_eeg[order][:, :, 5:60].reshape(N_TEST, -1))
    S = M @ P.T / M.shape[1]
    ranked = video_ids[np.argsort(-S, axis=1)]
    truth = gt[order]
    top1 = (ranked[:, 0] == truth).mean()
    top5 = (ranked[:, :5] == truth[:, None]).any(axis=1).mean()
    ranks = (-S).argsort(1).argsort(1)[np.arange(N_TEST), np.arange(N_TEST)] + 1
    results[s] = (top1, top5, np.median(ranks))
    print(f'sub-{s:02d}:  top-1 {top1:.4f}   top-5 {top5:.4f}   median rank {np.median(ranks):.1f}')

mt1 = np.mean([results[s][0] for s in DEV_SUBJECTS])
mt5 = np.mean([results[s][1] for s in DEV_SUBJECTS])
print(f'\nMEAN over dev subjects:  top-1 {mt1:.4f}   top-5 {mt5:.4f}')
print('\nReference (sub-01):')
print('  Option A (BERG direct):        top-1 0.0196  top-5 0.0784')
print('  AlexNet (notebook 02):         top-1 0.0392  top-5 0.1176')
print('  Option B (in silico, nb 04):   top-1 0.0490  top-5 0.2157')

sub-01:  top-1 0.0490   top-5 0.1275   median rank 25.5
sub-02:  top-1 0.0588   top-5 0.1765   median rank 33.5
sub-03:  top-1 0.0294   top-5 0.1471   median rank 34.5
sub-04:  top-1 0.0196   top-5 0.0980   median rank 42.0

MEAN over dev subjects:  top-1 0.0392   top-5 0.1373

Reference (sub-01):
  Option A (BERG direct):        top-1 0.0196  top-5 0.0784
  AlexNet (notebook 02):         top-1 0.0392  top-5 0.1176
  Option B (in silico, nb 04):   top-1 0.0490  top-5 0.2157


## Notes
- Feature = BERG ViT-B/32 **last-layer CLS token** (768-d), averaged over 8
  frames. Switch `VIT_LAYER` / `POOL` to experiment (earlier layers, mean-pool).
- One encoder is trained on pooled pairs from sub-01..04, so it learns a
  subject-agnostic ViT->EEG map. Matching is per subject against that subject's
  own test EEG.
- If this beats notebook 04's 0.0490 / 0.2157, it is the new best configuration
  and the submission notebook (05) can be updated to use the ViT feature +
  pooled training (while still ranking sub-05/06 in ORIGINAL order with no
  cross-subject test templates).